<a href="https://colab.research.google.com/github/lifan149/notes/blob/main/fastai/Practical-Deep-Learning-for-Coders/license_plate_recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
! pip install -Uqq fastbook
import fastbook
fastbook.setup_book()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 53.1 MB/s eta 0:00:00
Mounted at /content/gdrive


In [6]:
import os

# 定义解压目标目录
# 目标目录和zip文件在同一个文件夹，unzip会自动将内容放在该目录下
# 所以，这里可以直接使用zip文件所在的目录作为目标目录
destination_dir = '/content/gdrive/MyDrive/DataSet/'

# # 定义你的ZIP文件在Google Drive上的路径
zip_file_path = destination_dir + 'CBLPRD-330k_v1.zip'


if not os.path.isfile(zip_file_path):
    print(f"文件 '{zip_file_path}' 不存在或不是一个文件。")
else:
    if not os.path.exists(destination_dir):
        print(f"目标目录 '{destination_dir}' 不存在，正在创建...")
        os.makedirs(destination_dir) # 创建目录，包括所有缺失的父目录
        print(f"目录 '{destination_dir}' 创建成功。")
    else:
        print(f"目标目录 '{destination_dir}' 已存在。")

    # 4. 执行解压操作
    print(f"正在将 '{zip_file_path}' 解压到 '{destination_dir}'...")
    # -q: 安静模式，不显示详细的解压过程
    # -d: 指定解压目录
    !unzip -q "$zip_file_path" -d "$destination_dir"

    print("解压完成！")

    # 5. 验证解压结果 (可选)
    # 列出目标目录下的内容，看看文件是否成功解压
    print(f"'{destination_dir}' 中的文件列表：")
    !ls -lh "$destination_dir"

directory_path = destination_dir + "CBLPRD-330k"
count = 0
# os.walk 在 Python 3.5+ 内部已经使用 os.scandir() 进行优化
for root, dirs, files in os.walk(directory_path):
    for filename in files:
        # 避免额外的 is_file() 检查，因为 os.walk 已经区分了文件和目录
        if filename.lower().endswith('.jpg'):
            count += 1

print(f"在 '{directory_path}' 目录下 (包括子目录) 找到 {count} 个 .jpg/.JPG 文件 (最快递归方式)。")



在 '/content/gdrive/MyDrive/DataSet/CBLPRD-330k' 目录下 (包括子目录) 找到 342110 个 .jpg/.JPG 文件 (最快递归方式)。


In [19]:
from fastai.vision.all import *
from fastbook import *


dataset_root_path = Path(destination_dir + 'CBLPRD-330k/')

### 定义字符集和映射

**为什么要定义字符集和映射**

在fastai教程识别熊（如灰熊、黑熊、泰迪熊）的例子中并没有定义字符集和映射，为什么车牌号识别需要？这是因为fastai 教程中训练识别熊的模型，使用的是 `CategoryBlock`，而车牌识别需要自定义字符映射，这是因为这两者的**任务类型和输出形式完全不同**。

### 熊分类模型 (Image Classification)

在识别熊（如灰熊、黑熊、泰迪熊）的例子中，这是一个典型的**图像分类**任务：

* **输入：** 一张图片。
* **输出：** 图片属于预定义类别中的**一个**（例如，`Grizzly` 或 `Black` 或 `Teddy`）。

回顾一下 Fastai 的 `DataBlock` 配置：

```python
bears = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=parent_label,
    item_tfms=Resize(128)
)
```

这里发生了什么：

1.  **`ImageBlock`：** 明确告诉 Fastai，输入是图像。
2.  **`CategoryBlock`：** 明确告诉 Fastai，输出是一个**离散的类别标签**。
3.  **`get_y=parent_label`：** Fastai 在这里非常聪明地利用了文件系统结构。如果你的图片是这样组织的：
    ```
    images/
    ├── grizzly/
    │   ├── img1.jpg
    │   └── img2.jpg
    └── black/
        ├── img3.jpg
        └── img4.jpg
    ```
    那么 `parent_label` 函数会**自动提取父目录的名称作为类别标签**。例如，`images/grizzly/img1.jpg` 的标签就是 `'grizzly'`。

**Fastai 自动处理了什么？**

当 `CategoryBlock` 接收到像 `'grizzly'`、`'black'` 这样的字符串标签时，Fastai 在内部会**自动**为你完成字符到索引的映射：

* 它会收集所有唯一的类别字符串（例如，`['grizzly', 'black', 'teddy']`）。
* 然后它会为这些类别创建一个内部的“词汇表”（`vocab`），将每个类别字符串映射到一个唯一的整数索引（例如，`'grizzly' -> 0`, `'black' -> 1`, `'teddy' -> 2`）。
* 在训练时，你的标签会被转换为这些整数索引。
* 在预测时，模型输出的是这些整数索引，Fastai 会再次使用其内部的 `vocab` 将索引映射回类别字符串。

所以，你没有手动定义字符集和映射，是因为 Fastai 的 `CategoryBlock` 模块已经为你**封装了**这个“字符串类别到整数索引”的映射过程，并将其命名为 `vocab`。

### 车牌识别模型 (Image to Sequence / OCR)

而车牌识别则是一个**图像到文本序列**的任务：

* **输入：** 一张图片。
* **输出：** 一个**变长的字符序列**（例如，"京N12345"），而不是一个单一的离散类别。

你的车牌号 `云F5MTG学` 并不是一个单一的类别，它是多个字符组成的序列。车牌号的数量是无限的（字符的组合），你不可能为每一个可能的车牌号都定义一个类别。

因此，我们需要：

1.  **定义所有可能的单个字符**（`'京', '沪', ..., '0', '1', ..., 'A', 'B', ...`）。
2.  **将每个字符映射到唯一的整数ID。**
3.  **将整个车牌字符串（如“湘QSM5FJ”）编码成这些字符ID的序列。**
4.  模型会预测一个概率分布序列，每个时间步的概率分布对应你定义的字符集。
5.  **CTC Loss** 需要知道这些字符ID，尤其是空白字符的ID。
6.  **解码过程**需要将模型输出的字符ID序列转换回可读的车牌字符串。

Fastai 的 `CategoryBlock` 适用于单个分类标签，**不直接支持变长序列的字符识别**。虽然 Fastai 有 `TextBlock` 用于文本任务，但那是针对文本数据的（输入是文本，输出也是文本，或者文本分类），不直接适用于图像到文本的场景。

所以，在车牌识别中，我们必须**手动**定义字符集（所有可能的单个字符）以及它们到索引的映射，并创建自定义的 `PlateLabelBlock` 来处理这种序列标签。

### 总结核心区别：

* **熊分类：** 是**单标签分类**。Fastai 的 `CategoryBlock` 自动处理类别字符串到整数索引的映射（`vocab`）。
* **车牌识别：** 是**序列生成/识别**。输出是变长的字符序列，需要手动定义所有可能**单个字符**的集合以及它们到索引的映射，因为这是一个更底层的字符级预测任务。

In [18]:
# 定义所有可能的车牌字符
# 请注意，这里我将你提供的所有字符放到了一个列表中，确保没有重复
# 并且通常会添加一个 'blank' 字符用于 CTC Loss
CHARACTERS = [
    '京', '沪', '津', '渝', '冀', '晋', '蒙', '辽', '吉', '黑',
    '苏', '浙', '皖', '闽', '赣', '鲁', '豫', '鄂', '湘', '粤',
    '桂', '琼', '川', '贵', '云', '藏', '陕', '甘', '青', '宁',
    '新', '港', '澳', '挂', '学', '领', '使', '临',
    '0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
    'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'J', 'K',
    'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'U', 'V',
    'W', 'X', 'Y', 'Z', 'I', 'O',
    '<blank>' # CTC Loss 需要一个空白字符
]

# enumerate()方法可以同时获取可迭代对象中元素的索引 (index) 和值 (value) 时
# 创建字符到索引的映射 (char_to_idx)
char_to_idx = {char: i for i, char in enumerate(CHARACTERS)}
print(f"char_to_idx的类型{type(char_to_idx)}值： {char_to_idx} ");

# 创建索引到字符的映射 (idx_to_char)
idx_to_char = {i: char for i, char in enumerate(CHARACTERS)}
print(f"idx_to_char的类型{type(idx_to_char)}值： {idx_to_char} ");


# 编码和解码函数
def encode_plate(plate_str):
    """将车牌字符串编码为索引序列."""
    return [char_to_idx[char] for char in plate_str]

def decode_plate(idx_seq):
    """将索引序列解码为车牌字符串."""
    # CTC 解码时可能会有重复字符和 blank，这里只是一个简单的解码
    # 实际 CTC 解码需要更复杂的逻辑，去除重复和空白字符
    return ''.join([idx_to_char[idx] for idx in idx_seq if idx != char_to_idx['<blank>']])

# 车牌类型（如果需要作为额外的分类任务，或者用于数据分析）
PLATE_TYPES = [
    '黑色车牌', '单层黄牌', '双层黄牌', '普通蓝牌', '拖拉机绿牌',
    '新能源大型车', '新能源小型车'
]
# 如果需要，也可以为车牌类型创建映射
plate_type_to_idx = {p_type: i for i, p_type in enumerate(PLATE_TYPES)}
print(f"plate_type_to_idx的类型{type(plate_type_to_idx)}值： {plate_type_to_idx} ");


char_to_idx的类型<class 'dict'>值： {'京': 0, '沪': 1, '津': 2, '渝': 3, '冀': 4, '晋': 5, '蒙': 6, '辽': 7, '吉': 8, '黑': 9, '苏': 10, '浙': 11, '皖': 12, '闽': 13, '赣': 14, '鲁': 15, '豫': 16, '鄂': 17, '湘': 18, '粤': 19, '桂': 20, '琼': 21, '川': 22, '贵': 23, '云': 24, '藏': 25, '陕': 26, '甘': 27, '青': 28, '宁': 29, '新': 30, '港': 31, '澳': 32, '挂': 33, '学': 34, '领': 35, '使': 36, '临': 37, '0': 38, '1': 39, '2': 40, '3': 41, '4': 42, '5': 43, '6': 44, '7': 45, '8': 46, '9': 47, 'A': 48, 'B': 49, 'C': 50, 'D': 51, 'E': 52, 'F': 53, 'G': 54, 'H': 55, 'J': 56, 'K': 57, 'L': 58, 'M': 59, 'N': 60, 'P': 61, 'Q': 62, 'R': 63, 'S': 64, 'T': 65, 'U': 66, 'V': 67, 'W': 68, 'X': 69, 'Y': 70, 'Z': 71, 'I': 72, 'O': 73, '<blank>': 74} 
idx_to_char的类型<class 'dict'>值： {0: '京', 1: '沪', 2: '津', 3: '渝', 4: '冀', 5: '晋', 6: '蒙', 7: '辽', 8: '吉', 9: '黑', 10: '苏', 11: '浙', 12: '皖', 13: '闽', 14: '赣', 15: '鲁', 16: '豫', 17: '鄂', 18: '湘', 19: '粤', 20: '桂', 21: '琼', 22: '川', 23: '贵', 24: '云', 25: '藏', 26: '陕', 27: '甘', 28: '青', 29: '宁', 30: 

### 解析数据集文件
加载 train.txt 和 val.txt 来获取图像路径和对应的车牌字符串

In [21]:
import pandas as pd

def load_data_from_txt(file_path, base_image_dir):
    """
    从数据文本文件加载图像路径和车牌标签。

    Args:
        file_path (Path): train.txt 或 val.txt 的路径。
        base_image_dir (Path): 图像文件所在的根目录（例如 CBLPRD-330k/）。

    Returns:
        pd.DataFrame: 包含 'image_path' 和 'plate_label' 的 DataFrame。
    """
    data = []
    with open(file_path) as fp:
      for line in fp.readlines():
        line_split = line.strip().split(" ")
        img_full_path = base_image_dir + line_split[0]
        data.appen({'image_path': img_full_path, 'plate_label': line_split[1]})

    return pd.DataFrame(data);



# 数据集文件的完整路径
# Python 标准库中的 pathlib 模块提供了一种面向对象的方式来处理文件系统路径
# pathlib.Path 对象与 / 运算符拼接时，无论路径末尾是否有 / (斜杠)，都完全没有关系**，都可以直接用 / 进行拼接
train_txt_path = dataset_root_path.parent / 'train.txt'
val_txt_path = dataset_root_path.parent / 'val.txt'

# 加载训练和验证数据
train_df = load_data_from_txt(train_txt_path, dataset_root_path);
val_df = load_data_from_txt(val_txt_path, dataset_root_path);

print(f"训练集大小: {len(train_df)}")
print(f"验证集大小: {len(val_df)}")
print("训练集前5行:\n", train_df.head())

训练集大小: 325005
验证集大小: 17105
训练集前5行:
                                                               image_path  \
0  /content/gdrive/MyDrive/DataSet/CBLPRD-330k/CBLPRD-330k/000272981.jpg   
1  /content/gdrive/MyDrive/DataSet/CBLPRD-330k/CBLPRD-330k/000204288.jpg   
2  /content/gdrive/MyDrive/DataSet/CBLPRD-330k/CBLPRD-330k/000390092.jpg   
3  /content/gdrive/MyDrive/DataSet/CBLPRD-330k/CBLPRD-330k/000461632.jpg   
4  /content/gdrive/MyDrive/DataSet/CBLPRD-330k/CBLPRD-330k/000278419.jpg   

  plate_label  
0    粤Z31632D  
1    藏CFF7440  
2     晋ZFSD44  
3    苏GDG8575  
4    冀Q18266D  
